# ISLR Ch 4.7 #4 (ISLRv2 4.8 #4)

**(a) p = 1**

If X is uniform on [0, 1] and we use observations within 10% of the range around the test point, the interval has length 0.1. Since X is uniform, the fraction of observations in an interval equals its length, so on average we use **10%** of the observations. (Strictly, test points within 0.05 of an edge have a truncated interval, which pulls the exact average down slightly to 9.75%. The standard answer ignores this edge effect.)

In [ ]:
# (a) fraction of observations used with p = 1
frac_a <- 0.1^1
frac_a

# exact average including edge effects: E[length of [x-0.05, x+0.05] ∩ [0,1]]
integrate(function(x) pmin(x + 0.05, 1) - pmax(x - 0.05, 0), 0, 1)$value

**(b) p = 2**

The two features are independent and uniform, so the fraction of observations in the square is the product of the side lengths: 0.1 × 0.1 = **1%** of the observations.

In [ ]:
# (b) fraction of observations used with p = 2
frac_b <- 0.1^2
frac_b

**(c) p = 100**

Following the same logic, the fraction is 0.1^100 = **10^-100**. This is effectively zero: no realistic dataset would contain even one observation in the neighborhood.

In [ ]:
# (c) fraction of observations used with p = 100
frac_c <- 0.1^100
frac_c

In [ ]:
# Simulation check: fraction of uniform points within +/-0.05 of a test point in every dimension
set.seed(1)
sim_frac <- function(p, n = 100000) {
  x0 <- runif(p, 0.05, 0.95)           # test point (interior, to avoid edge effects)
  X  <- matrix(runif(n * p), n, p)
  mean(rowSums(abs(sweep(X, 2, x0)) <= 0.05) == p)
}
sapply(c(1, 2, 3, 5, 10), sim_frac)
0.1^c(1, 2, 3, 5, 10)

**(d) Why this is a problem for KNN**

The fraction of training data that is "near" a test point shrinks exponentially with p: 10% when p = 1, 1% when p = 2, and 10^-100 when p = 100. With p large, essentially no training observations lie within 10% of the test point on every feature. KNN's K "nearest" neighbors are then actually far away from the test point, so they are not local and are poor predictors of its response. The prediction is no longer based on similar observations, which degrades KNN's performance (high bias) unless n is astronomically large.

**(e) Hypercube containing 10% of the training observations**

A hypercube centered on the test point with side length s contains a fraction s^p of the observations (uniform data). Setting s^p = 0.1 gives **s = 0.1^(1/p)**.

- p = 1: s = 0.1
- p = 2: s = 0.1^(1/2) ≈ 0.316
- p = 100: s = 0.1^(1/100) ≈ 0.977

In [ ]:
# (e) side length of hypercube containing 10% of the observations
p <- c(1, 2, 100)
data.frame(p = p, side_length = 0.1^(1 / p))

For p = 1 the neighborhood is just 10% of the range of X, but for p = 2 it must span about 32% of each feature's range, and for p = 100 it must span about 98% of each feature's range. To capture even 10% of the data, the "neighborhood" has to cover almost the entire range of every feature, so it is not local at all. This is the curse of dimensionality: in high dimensions, a neighborhood that is truly local contains almost no data, and a neighborhood that contains enough data is no longer local.